# 21 - Delta Time Travel Demo

Queries the previous Delta versions of one Silver and one Gold
table using `versionAsOf`, displays samples, and persists the
version/count evidence before any real VACUUM execution.


In [ ]:
%run ./05_bronze_common


In [ ]:
import uuid

from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql import types as T


dbutils.widgets.text("catalog", "abc_retail", "Target catalog")
dbutils.widgets.text("job_run_id", "manual", "Notebook or job run ID")

catalog = validate_identifier(dbutils.widgets.get("catalog").strip(), "catalog")
job_run_id = dbutils.widgets.get("job_run_id").strip()
spark.conf.set("spark.sql.session.timeZone", "UTC")

pipeline_name = "time_travel_demo"
pipeline_run_id = str(uuid.uuid4())
start_ts = datetime.now(timezone.utc)
evidence_table = f"{catalog}.control.time_travel_evidence"
tables = [
    f"{catalog}.silver.silver_events",
    f"{catalog}.gold.gold_daily_funnel",
]

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS `{catalog}`.`control`.`time_travel_evidence` (
        table_name STRING NOT NULL,
        observed_ts TIMESTAMP NOT NULL,
        current_version BIGINT NOT NULL,
        previous_version BIGINT NOT NULL,
        previous_version_timestamp TIMESTAMP NOT NULL,
        current_row_count BIGINT NOT NULL,
        previous_row_count BIGINT NOT NULL,
        pipeline_run_id STRING NOT NULL
    )
    USING DELTA
    TBLPROPERTIES (
        'comment' = 'Persisted evidence that earlier Delta versions remain queryable'
    )
    """
)

upsert_audit(
    catalog, pipeline_run_id, pipeline_name, "version_as_of", start_ts,
    "RUNNING", tables, job_run_id,
)

schema = T.StructType(
    [
        T.StructField("table_name", T.StringType(), False),
        T.StructField("observed_ts", T.TimestampType(), False),
        T.StructField("current_version", T.LongType(), False),
        T.StructField("previous_version", T.LongType(), False),
        T.StructField("previous_version_timestamp", T.TimestampType(), False),
        T.StructField("current_row_count", T.LongType(), False),
        T.StructField("previous_row_count", T.LongType(), False),
        T.StructField("pipeline_run_id", T.StringType(), False),
    ]
)

try:
    observed_ts = datetime.now(timezone.utc)
    evidence = []
    total_rows_read = 0

    for table in tables:
        versions = (
            spark.sql(f"DESCRIBE HISTORY {table}")
            .select("version", "timestamp")
            .orderBy(F.col("version").desc())
            .limit(2)
            .collect()
        )
        if len(versions) < 2:
            raise RuntimeError(
                f"{table} needs at least two Delta versions for Time Travel"
            )

        current_version = int(versions[0].version)
        previous_version = int(versions[1].version)
        current_df = spark.table(table)
        quoted = ".".join(f"`{part}`" for part in table.split("."))
        previous_df = spark.sql(
            f"SELECT * FROM {quoted} VERSION AS OF {previous_version}"
        )
        current_count = current_df.count()
        previous_count = previous_df.count()
        total_rows_read += current_count + previous_count

        evidence.append(
            (
                table,
                observed_ts,
                current_version,
                previous_version,
                versions[1].timestamp,
                current_count,
                previous_count,
                pipeline_run_id,
            )
        )
        print(
            f"{table}: current version={current_version}, "
            f"previous version={previous_version}, "
            f"counts={current_count}/{previous_count}"
        )
        previous_df.limit(10).display()

    evidence_df = spark.createDataFrame(evidence, schema)
    evidence_df.write.format("delta").mode("append").saveAsTable(evidence_table)
    upsert_audit(
        catalog, pipeline_run_id, pipeline_name, "version_as_of", start_ts,
        "SUCCESS", tables, job_run_id,
        end_ts=datetime.now(timezone.utc), rows_read=total_rows_read,
        rows_written=len(evidence), rows_quarantined=0,
    )
except Exception as exc:
    upsert_audit(
        catalog, pipeline_run_id, pipeline_name, "version_as_of", start_ts,
        "FAILED", tables, job_run_id,
        end_ts=datetime.now(timezone.utc), error_message=str(exc)[:4000],
    )
    raise

spark.table(evidence_table).orderBy(F.col("observed_ts").desc()).display()
